<a href="https://colab.research.google.com/github/nisaa783/scikit-learn-cookbook_Anisa-Khasanah/blob/main/Chapter_13_Deploying_Models_in_Production.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Pendahuluan dan Transisi Model ke Produksi
Melatih model di notebook eksperimental hanyalah salah satu bagian dari siklus hidup machine learning. Agar memberikan nilai praktis, model beserta seluruh tahapan pra-pemrosesannya harus dapat disimpan, dimuat kembali, dan digunakan secara konsisten pada data baru di lingkungan produksi atau layanan aplikasi.

Tantangan utama saat memindahkan model ke produksi adalah memastikan bahwa transformasi fitur, urutan langkah pemrosesan, versi pustaka, dan format input data baru identik dengan saat model dilatih. Menyimpan objek Pipeline lengkap (bukan hanya estimator akhirnya) membantu menjaga konsistensi alur kerja tersebut.

## 2. Serialisasi Model dan Validasi Konsistensi Produksi
Untuk menyimpan objek Python seperti Pipeline atau model terlatih ke dalam berkas di disk dan memuatnya kembali di proses atau sesi lain, pustaka joblib sering dipilih karena efisien untuk objek yang memuat array numerik besar, sementara modul bawaan pickle juga umum digunakan untuk serialisasi objek Python.

Setelah berkas model dimuat kembali, langkah penting sebelum menggunakannya untuk prediksi operasional adalah melakukan verifikasi kesetaraan keluaran (parity check) dengan membandingkan hasil prediksi model yang baru dimuat terhadap hasil prediksi model sebelum disimpan pada data uji yang sama. Selain itu, mengecek metadata seperti versi scikit-learn dan menangani skema fitur masukan membantu mencegah galat tak terduga saat data baru masuk.

Konsep & Rumus Matematika:Pada tahap inference di produksi, model terlatih memetakan vektor fitur masukan $x \in \mathbb{R}^d$ setelah melalui fungsi transformasi pipeline $\Phi(\cdot)$ ke ruang keluaran prediksi $\hat{y}$:$$\hat{y} = f_{\theta}\big(\Phi(x)\big)$$di mana $\Phi(x)$ merepresentasikan transformasi fitur yang parameter-nya telah ditetapkan saat pelatihan (misalnya mean dan scale pada penskalaan), dan $f_{\theta}$ adalah fungsi prediksi model dengan parameter terlatih $\theta$. Untuk memantau stabilitas prediksi pada batch data baru berukuran $m$, rata-rata skor probabilitas kelas positif $\bar{p}$ dapat dihitung sebagai:$$\bar{p} = \frac{1}{m} \sum_{i=1}^{m} P(y_i = 1 \mid x_i)$$

## 3. Contoh Program Penerapan 1: Menyimpan dan Memuat Kembali Pipeline dengan joblib
Contoh program pertama ini melatih sebuah Pipeline klasifikasi, menyimpannya ke berkas menggunakan joblib.dump, memuatnya kembali menggunakan joblib.load, dan memastikan hasil prediksinya identik dengan model awal.


In [1]:
from pathlib import Path
import joblib
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Memuat dataset dan membagi data latih serta data uji
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Melatih Pipeline lengkap (preprocessing + estimator)
model_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42))
])
model_pipeline.fit(X_train, y_train)

# Prediksi awal sebelum serialisasi
pred_before = model_pipeline.predict(X_test)
prob_before = model_pipeline.predict_proba(X_test)[:, 1]

# 3. Menyimpan model ke disk menggunakan joblib
artifact_path = Path("breast_cancer_rf_pipeline.joblib")
joblib.dump(model_pipeline, artifact_path)
print(f"Artefak model disimpan ke: {artifact_path.resolve()}")
print(f"Ukuran berkas: {artifact_path.stat().st_size} bytes")

# 4. Memuat kembali model dan memverifikasi konsistensi prediksi
loaded_pipeline = joblib.load(artifact_path)
pred_after = loaded_pipeline.predict(X_test)
prob_after = loaded_pipeline.predict_proba(X_test)[:, 1]

is_identical = np.array_equal(pred_before, pred_after)
max_prob_diff = np.max(np.abs(prob_before - prob_after))

print(f"Hasil prediksi identik 100%: {is_identical}")
print(f"Selisih maksimum probabilitas: {max_prob_diff:.2e}")
print(f"Akurasi pada data uji: {loaded_pipeline.score(X_test, y_test):.4f}")

Artefak model disimpan ke: /content/breast_cancer_rf_pipeline.joblib
Ukuran berkas: 325506 bytes
Hasil prediksi identik 100%: True
Selisih maksimum probabilitas: 0.00e+00
Akurasi pada data uji: 0.9561


## 4. Contoh Program Penerapan 2: Membungkus Inference Service dengan Validasi Input dan Metadata
Contoh program kedua ini mengemas artefak model bersama metadata (seperti nama fitur dan versi pustaka) serta membuat fungsi inference sederhana yang memeriksa kelengkapan fitur masukan sebelum menghasilkan prediksi kelas dan probabilitas.

In [2]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Melatih pipeline dan menyusun bundel artefak produksi
data = load_breast_cancer()
X_df = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=42, stratify=y
)

clf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42))
])
clf_pipeline.fit(X_train, y_train)

bundle = {
    "pipeline": clf_pipeline,
    "feature_names": list(data.feature_names),
    "target_names": list(data.target_names),
    "sklearn_version": sklearn.__version__
}

bundle_path = Path("production_model_bundle.joblib")
joblib.dump(bundle, bundle_path)

# 2. Fungsi layanan prediksi dengan validasi skema fitur
def predict_records(records_df: pd.DataFrame, model_file: Path):
    loaded_bundle = joblib.load(model_file)
    expected_cols = loaded_bundle["feature_names"]
    target_names = loaded_bundle["target_names"]
    pipe = loaded_bundle["pipeline"]

    missing_cols = [col for col in expected_cols if col not in records_df.columns]
    if missing_cols:
        raise ValueError(f"Kolom fitur berikut hilang pada data input: {missing_cols}")

    ordered_df = records_df[expected_cols]
    pred_ids = pipe.predict(ordered_df)
    pred_probs = pipe.predict_proba(ordered_df)

    results = []
    for idx, (cls_id, probs) in enumerate(zip(pred_ids, pred_probs)):
        results.append({
            "record_index": idx,
            "predicted_class_id": int(cls_id),
            "predicted_label": target_names[int(cls_id)],
            "confidence": float(np.max(probs))
        })
    return results

# 3. Uji fungsi prediksi pada 3 sampel data uji
sample_batch = X_test.iloc[:3].copy()
output = predict_records(sample_batch, bundle_path)

print(f"Versi scikit-learn saat pelatihan: {bundle['sklearn_version']}")
for item in output:
    print(item)

Versi scikit-learn saat pelatihan: 1.6.1
{'record_index': 0, 'predicted_class_id': 0, 'predicted_label': np.str_('malignant'), 'confidence': 0.9999999411175814}
{'record_index': 1, 'predicted_class_id': 1, 'predicted_label': np.str_('benign'), 'confidence': 0.9999886648327754}
{'record_index': 2, 'predicted_class_id': 0, 'predicted_label': np.str_('malignant'), 'confidence': 0.9935891753788668}
